In [ ]:
# =============================================================================
# 🚀 K-MHaS (Korean Hate Speech Detection) 초급 실습 스크립트
# 🌟 AI 코딩 튜터의 친절한 가이드가 함께합니다! 🌟
# =============================================================================
# 데이터셋 제목: jeanlee/kmhas_korean_hate_speech
# 대략적인 의미: 한국어 혐오 발언 감지 데이터셋
# 설명: 이 데이터셋은 한국어 텍스트가 특정 기준(성별, 인종, 정치 등)에 따라 혐오 발언인지 분류하는 데 사용됩니다.
# 💡 오늘 목표: 딥러닝 모델을 돌려보지 않고도, 데이터 구조와 레이블 시스템을 파헤치며
#       '이 데이터셋이 어떤 정보를 담고 있는지' 창의적으로 탐험하는 것이 목표입니다!

import random
from datasets import load_dataset, Dataset

# --- 설정값 ---
DATASET_NAME = "jeanlee/kmhas_korean_hate_speech"
SAMPLE_COUNT = 50  # 분석할 샘플의 개수 (너무 많으면 시간이 오래 걸리니까 적당히!)

# 레이블 이름 매핑 (데이터셋의 클래스 구조를 이해하는 것이 핵심!)
LABEL_NAMES = [
    "origin",
    "physical",
    "politics",
    "profanity",
    "age",
    "gender",
    "race",
    "religion",
    "not_hate_speech"
]
# -----------------------------------------------------------------------------

print("🌟 환영합니다! 튜터 AI와 함께 한국어 혐오 발언 감지 데이터셋 탐험을 시작해 봅시다. 🚀")

# 1. 데이터 로드 (Stream/Non-Stream 안전 장치 마련하기)
try:
    # 🥇 1차 시도: 스트리밍 모드 (최대한 빠르고 가볍게!)
    # 주의: 대용량 데이터셋의 경우 메모리 폭발을 막아줍니다.
    raw_dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("\n✅ [Step 1] 데이터셋 로드 성공! 스트리밍 모드로 데이터셋 연결 준비 완료.")
except Exception as e:
    print(f"\n⚠️ [Step 1] 스트리밍 로드 실패 ({e}). 일반 모드로 전환하여 소량 로드합니다.")
    # 🥈 2차 시도: 스트리밍 실패 시, 테스트 셋의 일부만 로드하여 진행합니다.
    raw_dataset = load_dataset(DATASET_NAME, split='train')

# 2. 샘플 데이터 준비 (전체 데이터 대신, 맛보기 50개만 골라 봅시다!)
# 스트리밍/비스트리밍 환경에 맞춰 데이터를 추출하는 가장 안전하고 쉬운 방법!
if hasattr(raw_dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)일 가능성이 높습니다.
    print(f"✨ {SAMPLE_COUNT}개의 샘플을 스트리밍 방식으로 추출할 준비를 합니다.")
    sampled_dataset_iterator = raw_dataset.take(SAMPLE_COUNT)
    # 메모리 효율을 위해 실제 리스트로 변환합니다. (실습에 적합한 패턴!)
    sample_data_list = list(sampled_dataset_iterator)
else:
    # 일반 Dataset 객체일 경우
    sample_data_list = list(raw_dataset.select(range(min(SAMPLE_COUNT, len(raw_dataset)))))


# 3. 데이터 탐험 함수 정의 (데이터 해석이 가장 재미있는 부분!)
def analyze_sample(sample):
    """하나의 샘플을 받아 레이블을 해석하여 혐오 발언 프로파일을 만듭니다."""
    text = sample['text']
    labels_sequence = sample['label']['feature']['names'] # 레이블 이름 목록 가져오기

    # Multi-label classification의 특성상, 어떤 레이블이 '활성화'되었는지 체크합니다.
    active_labels = []
    for i in range(len(labels_sequence)):
        # labels_sequence[i]는 ClassLabel 객체이며, 값이 True면 활성화된 것입니다.
        if labels_sequence[i].is_true: # 실제로는 True/False로 체크됨
            active_labels.append(labels_sequence[i])

    return active_labels

# 4. 실행 및 분석 (학습자에게 가장 흥미로운 실습!)
print("\n" + "="*80)
print(f"✨ [Step 2] 데이터 분석 시작: 상위 {len(sample_data_list)}개 샘플 탐험하기!")
print("✨ 이 데이터를 보고, 어떤 유형의 혐오 발언이 가장 흔한지 예측해 보세요!")
print("="*80)

print("\n--- 🌈 '랜덤 혐오 프로파일러' 실행 결과 ---")
print(f"총 {len(sample_data_list)}개 샘플에 대한 분석을 시작합니다.")

# 분석 결과를 저장할 리스트
profile_results = []

for i, sample in enumerate(sample_data_list):
    try:
        # 레이블 순서에 따라 개별 레이블의 True/False 값을 가져옵니다.
        labels_list = []
        for label_idx in range(len(LABEL_NAMES)):
            # labels_sequence[label_idx]에 True/False가 들어 있습니다.
            is_activated = sample['label']['feature'].names[label_idx].is_true
            labels_list.append(is_activated)

        # 활성화된 레이블 이름을 해석합니다.
        active_labels = [LABEL_NAMES[j] for j, is_active in enumerate(labels_list) if is_active]
        
        # 📜 예쁜 형식으로 출력
        print(f"\n[{i+1:03}] 💬 원문 텍스트: {sample['text'][:60]}...")
        print(f"   -> 🔎 감지된 혐오 프로파일 (활성화된 레이블): {', '.join(active_labels)}")
        
        profile_results.append({
            'text': sample['text'],
            'labels': active_labels
        })

    except Exception as e:
        # 데이터 형식 오류가 발생해도 스크립트가 멈추지 않도록 방어 코드입니다.
        print(f"🚨 [경고] {i+1}번째 샘플에서 데이터 처리 오류 발생: {e}")

# 5. 종합 분석 (통계적 인사이트 끌어내기)
print("\n" + "="*80)
print("✨ [Step 3] 종합 분석: 어떤 유형이 가장 '잘 잡히는'가? (빈도수 계산)")
print("="*80)

# 카운팅 딕셔너리 초기화
label_counts = {name: 0 for name in LABEL_NAMES}
total_samples = len(profile_results)

# 활성화된 레이블의 총 빈도수를 계산합니다.
for profile in profile_results:
    for label in profile['labels']:
        if label in label_counts:
            label_counts[label] += 1

# 결과를 보기 좋게 출력
print(f"📊 분석된 총 샘플 수: {total_samples}개")
print("--- 레이블별 활성화 빈도수 (총 혐오 발언 '표식' 갯수) ---")

# not_hate_speech는 항상 True가 아닐 것이므로 제외하고 분석합니다.
for label, count in label_counts.items():
    if label != "not_hate_speech":
        print(f"🔹 {label:<15}: {count:,} 건 (전체 샘플 대비 {count/total_samples * 100:.1f}%)")

print("\n🎉 **튜터의 마지막 조언!** 🎉")
print("이 분석을 통해, 이 데이터셋은 단순히 텍스트만 아니라, 누가(gender), 무엇에 대해(politics), 왜(origin) 혐오하는지 분류할 수 있는 아주 풍부한 정보를 담고 있답니다!")
print("이제 이 지식을 바탕으로 '어떤 혐오 발언 탐지기가 가장 필요한지' 아이디어를 떠올려 보세요! 👏")